# 04 — Analista conversacional — LockBit Panel

Interfaz de consulta en lenguaje natural sobre la base de datos.  
El LLM recibe el esquema de los DataFrames, genera código pandas para responder  
y el notebook lo ejecuta mostrando código + resultado.

**Uso**: edita la celda de pregunta y ejecuta. Puedes hacer preguntas en español o inglés.

## Setup — ejecutar una vez al abrir el notebook

In [ ]:
import re
import json
import pandas as pd
import ollama
from pathlib import Path
from IPython.display import display, Markdown

PROCESSED = Path('../data/processed')
MODEL     = 'qwen3.5'

# Cargar tablas (chats_classified si existe, si no chats base)
users    = pd.read_parquet(PROCESSED / 'users.parquet')
clients  = pd.read_parquet(PROCESSED / 'clients.parquet')
builds   = pd.read_parquet(PROCESSED / 'builds.parquet')
invites  = pd.read_parquet(PROCESSED / 'invites.parquet')
btc      = pd.read_parquet(PROCESSED / 'btc_addresses.parquet')

chats_path = PROCESSED / 'chats_classified.parquet'
chats = pd.read_parquet(chats_path if chats_path.exists() else PROCESSED / 'chats.parquet')

DATAFRAMES = {
    'users': users,
    'clients': clients,
    'chats': chats,
    'builds': builds,
    'invites': invites,
    'btc_addresses': btc,
}

print('Tablas cargadas:')
for name, df in DATAFRAMES.items():
    print(f'  {name:20s} {len(df):6,} filas x {len(df.columns)} cols')

In [2]:
def _build_schema_context() -> str:
    """Construye el contexto de esquema para el LLM."""
    lines = ["Available pandas DataFrames (LockBit ransomware panel DB, Dec 2024–Apr 2025):\n"]
    for name, df in DATAFRAMES.items():
        lines.append(f"  {name} ({len(df):,} rows)")
        col_info = ', '.join(
            f"{c} [{df[c].dtype}]" for c in df.columns
        )
        lines.append(f"    columns: {col_info}")
        # Muestra 1 fila representativa como referencia de valores
        sample = df.dropna(how='all').head(1)
        if len(sample):
            for col in sample.columns[:8]:   # solo primeras 8 cols para no saturar
                val = sample[col].iloc[0]
                val_str = str(val)[:60].replace('\n', ' ')
                lines.append(f"    {col}: {val_str}")
        lines.append('')
    return '\n'.join(lines)

SCHEMA_CONTEXT = _build_schema_context()

SYSTEM_PROMPT = f"""You are a forensic data analyst working with the leaked LockBit ransomware panel database.
Answer questions by writing Python/pandas code that queries the available DataFrames.

{SCHEMA_CONTEXT}
Key facts:
- `clients` = compromised victim companies. `paid_commission=1` means they paid the ransom.
- `chats` = negotiations between LockBit operators (flag=1) and victims (flag=0).
- `chats.phase` (if present) = negotiation phase classified by LLM.
- `users` = LockBit operators/affiliates. `is_admin=1` = admin.
- `builds` = malware builds created by operators. `type` = LockBit variant (25/30/40/46/50).
- `invites` = affiliate recruitment codes. `status=50` = accepted.
- Timestamps are UTC. Unix timestamps in `clients.date_first`, `clients.date_last`, `builds.date`.
- `clients.advid` and `chats.advid` link to `users.id`.
- `clients.build_id` links to `builds.id`.

Rules:
- Write clean, runnable Python using pandas. Do NOT import anything — all libraries are already in scope.
- Store your final answer in a variable called `result`.
- `result` can be a DataFrame, Series, scalar, dict, or string.
- Add brief inline comments.
- If the question cannot be answered with the available data, say so clearly.

Always wrap your code in a ```python ... ``` block."""

# Historial de conversación (se acumula entre llamadas en la misma sesión)
_history = []

print('Sistema de consulta listo.')
print(f'Modelo: {MODEL}')
print('Usa ask("tu pregunta") para consultar.')

Sistema de consulta listo.
Modelo: qwen2.5:14b
Usa ask("tu pregunta") para consultar.


In [3]:
def ask(question: str, show_code: bool = True, memory: bool = True):
    """
    Envía una pregunta al LLM, ejecuta el código generado y muestra el resultado.

    Args:
        question   : pregunta en lenguaje natural (español o inglés)
        show_code  : mostrar el código generado antes del resultado (default True)
        memory     : incluir historial de la sesión para preguntas de seguimiento
    """
    global _history

    messages = [{'role': 'system', 'content': SYSTEM_PROMPT}]
    if memory:
        messages += _history
    messages.append({'role': 'user', 'content': question})

    resp = ollama.chat(
        model=MODEL,
        messages=messages,
        options={'temperature': 0.1}
    )
    answer = resp.message.content.strip()

    # Guardar en historial
    if memory:
        _history.append({'role': 'user', 'content': question})
        _history.append({'role': 'assistant', 'content': answer})
        # Limitar historial para no saturar contexto
        if len(_history) > 20:
            _history = _history[-20:]

    # Extraer bloque de código
    code_match = re.search(r'```python\n(.*?)```', answer, re.DOTALL)

    if code_match:
        code = code_match.group(1).strip()

        if show_code:
            display(Markdown(f'**Código generado:**\n```python\n{code}\n```'))

        # Ejecutar en namespace con acceso a todos los DataFrames
        namespace = {**DATAFRAMES, 'pd': pd, 'result': None}
        try:
            exec(code, namespace)
            result = namespace.get('result')
            display(Markdown('**Resultado:**'))
            if result is not None:
                display(result)
            else:
                print('(sin resultado — revisa el código generado)')
        except Exception as e:
            print(f'Error al ejecutar el código: {e}')
            display(Markdown('**Respuesta sin código:**'))
            display(Markdown(answer))
    else:
        # Respuesta directa sin código
        display(Markdown(answer))


def reset_memory():
    """Borra el historial de conversación para empezar de cero."""
    global _history
    _history = []
    print('Historial borrado.')

---
## Preguntas de ejemplo

Edita cualquier celda y ejecuta. Puedes añadir celdas nuevas con `B`.

In [ ]:
ask("¿Cuántas víctimas hay en total y cuántas pagaron el rescate?")

In [4]:
ask("¿Qué operador tiene más víctimas asignadas? Muestra su login y número de víctimas")

**Código generado:**
```python
import pandas as pd

# Join users and clients based on advid linking operators to their victims
operator_victims = pd.merge(users, clients, left_on='id', right_on='advid')

# Group by operator and count the number of victims each has
victim_counts = operator_victims.groupby('login').size().reset_index(name='num_victims')

# Find the operator with the most victims
result = victim_counts.loc[victim_counts['num_victims'].idxmax(), ['login', 'num_victims']]
```

**Resultado:**

login          Christopher
num_victims             44
Name: 8, dtype: object

In [ ]:
ask("¿Cuál fue el mes con más nuevas víctimas comprometidas?")

In [ ]:
ask("Muestra las víctimas que pagaron: su ID, operador asignado, y días que tardaron en pagar desde el primer contacto")

In [ ]:
ask("¿Cuántos builds de malware creó cada operador? Muestra los 10 más activos con su login")

In [5]:
ask("¿Cuál es la variante de LockBit más usada (campo type)? ¿Ha cambiado con el tiempo?")

**Código generado:**
```python
import pandas as pd

# Group builds by type and count occurrences to find the most used variant
variant_counts = builds['type'].value_counts().reset_index()
variant_counts.columns = ['type', 'count']

# Determine the most used variant
most_used_variant = variant_counts.loc[variant_counts['count'].idxmax(), 'type']
result_most_used = most_used_variant

# Check if there's a change over time by grouping builds by date and type
time_variant_change = pd.merge(builds, variant_counts, on='type')
time_variant_change['date'] = pd.to_datetime(time_variant_change['date'])
time_variant_change.sort_values(by=['date'], inplace=True)

# Group by date and most used variant to see if there's a change over time
variant_over_time = time_variant_change.groupby(['date', 'type']).size().reset_index(name='counts')

# Check for changes in the most common variant over time
changes_in_most_used = variant_over_time[variant_over_time['type'] != most_used_variant]

result = {
    'most_used_variant': result_most_used,
    'has_changed_over_time': not changes_in_most_used.empty
}
```

**Resultado:**

{'most_used_variant': np.int64(30), 'has_changed_over_time': True}

In [ ]:
ask("De las negociaciones de chat, ¿cuánto tiempo median (en horas) entre el primer y último mensaje por víctima?")

---
## Tu turno — edita esta celda con tu pregunta

In [6]:
ask("¿Cuál es la estructura de la base de datos?")

**Código generado:**
```python
structure = {
    'users': [
        {'column': 'id', 'type': 'int64'},
        {'column': 'parentid', 'type': 'int64'},
        {'column': 'login', 'type': 'str'},
        {'column': 'password', 'type': 'str'},
        {'column': 'is_admin', 'type': 'int64'},
        {'column': 'level', 'type': 'int64'},
        {'column': 'session_id', 'type': 'str'},
        # Otros campos
    ],
    'clients': [
        {'column': 'id', 'type': 'int64'},
        {'column': 'important', 'type': 'int64'},
        {'column': 'advid', 'type': 'int64'},
        {'column': 'master_pubkey', 'type': 'str'},
        {'column': 'session_key', 'type': 'str'},
        # Otros campos
    ],
    'chats': [
        {'column': 'id', 'type': 'int64'},
        {'column': 'advid_owner', 'type': 'int64'},
        {'column': 'advid', 'type': 'int64'},
        {'column': 'clientid', 'type': 'int64'},
        # Otros campos
    ],
    'builds': [
        {'column': 'id', 'type': 'int64'},
        {'column': 'parent_id', 'type': 'int64'},
        {'column': 'status', 'type': 'int64'},
        {'column': 'decription_id', 'type': 'str'},
        # Otros campos
    ],
    'invites': [
        {'column': 'id', 'type': 'int64'},
        {'column': 'invite', 'type': 'str'},
        {'column': 'order', 'type': 'object'},
        {'column': 'btc_wallet', 'type': 'str'},
        # Otros campos
    ],
    'btc_addresses': [
        {'column': 'id', 'type': 'int64'},
        {'column': 'type', 'type': 'int64'},
        {'column': 'target_id', 'type': 'int64'},
        {'column': 'advid', 'type': 'int64'},
        # Otros campos
    ]
}

result = structure
```

**Resultado:**

{'users': [{'column': 'id', 'type': 'int64'},
  {'column': 'parentid', 'type': 'int64'},
  {'column': 'login', 'type': 'str'},
  {'column': 'password', 'type': 'str'},
  {'column': 'is_admin', 'type': 'int64'},
  {'column': 'level', 'type': 'int64'},
  {'column': 'session_id', 'type': 'str'}],
 'clients': [{'column': 'id', 'type': 'int64'},
  {'column': 'important', 'type': 'int64'},
  {'column': 'advid', 'type': 'int64'},
  {'column': 'master_pubkey', 'type': 'str'},
  {'column': 'session_key', 'type': 'str'}],
 'chats': [{'column': 'id', 'type': 'int64'},
  {'column': 'advid_owner', 'type': 'int64'},
  {'column': 'advid', 'type': 'int64'},
  {'column': 'clientid', 'type': 'int64'}],
 'builds': [{'column': 'id', 'type': 'int64'},
  {'column': 'parent_id', 'type': 'int64'},
  {'column': 'status', 'type': 'int64'},
  {'column': 'decription_id', 'type': 'str'}],
 'invites': [{'column': 'id', 'type': 'int64'},
  {'column': 'invite', 'type': 'str'},
  {'column': 'order', 'type': 'object'},

---
## Preguntas de seguimiento (el LLM recuerda el contexto de la sesión)

Las preguntas encadenadas funcionan porque el historial se acumula.  
Usa `reset_memory()` para empezar una línea de preguntas nueva.

In [ ]:
# Ejemplo de pregunta encadenada:
# ask("¿Qué operador tiene más víctimas?")
# ask("¿Cuántos de sus builds son de tipo 30?")  # el LLM sabe a qué operador te refieres

# reset_memory()  # descomenta para borrar el historial